# Results: quantized, pruned autoencoder anomaly trigger

Reads the CSVs and plots written by the pipeline (see README for the run order). Every cell prints a note instead of failing if its input file is missing. Nothing here re-trains anything.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
ROOT = Path.cwd()
if not (ROOT / "results").exists() and (ROOT.parent / "results").exists():
    ROOT = ROOT.parent
R = ROOT / "results"

def load(name, folder=R):
    p = folder / name
    if not p.exists():
        print("missing:", p.relative_to(ROOT))
        return None
    return pd.read_csv(p)

## 1. Data and the trivial detector
Trivial score = mean of x² of the standardized input, i.e. the reconstruction error of an all-zero output. Any autoencoder has to beat it.

In [ ]:
meta_p, npz_p = ROOT / "data/processed/meta.json", ROOT / "data/processed/dataset.npz"
triv = None
if meta_p.exists() and npz_p.exists():
    from sklearn.metrics import roc_auc_score
    meta = json.loads(meta_p.read_text())
    print("class counts:", meta["class_counts"])
    print("QCD split sizes:", meta["split_sizes"])
    z = np.load(npz_p)
    neg = (z["X_test_qcd"] ** 2).mean(axis=1)
    pos = {k[len("X_anom_"):]: (z[k] ** 2).mean(axis=1) for k in z.files if k.startswith("X_anom_")}
    def _auc(p):
        return roc_auc_score(np.r_[np.zeros(len(neg)), np.ones(len(p))], np.r_[neg, p])
    triv = {k: _auc(v) for k, v in pos.items()}
    triv["pooled"] = _auc(np.concatenate(list(pos.values())))
    print("trivial detector AUC:", {k: round(v, 3) for k, v in triv.items()})
else:
    print("data/processed not found; run python -m src.prepare_dataset")

## 2. FP32 baselines
Single-seed results from Phase 2 (all five architectures tried), then the 5-seed spread of the three swept architectures.

In [ ]:
b = load("fp32_baseline_all.csv")
if b is None:
    b = load("fp32_baseline.csv")
if b is not None:
    display(b.filter(regex="arch|n_params|epochs_run|best_val|^auc_").round(3))
runs = load("sweep_runs.csv")
if runs is not None:
    fp = runs.drop_duplicates(["arch", "seed"])
    cols = [c for c in fp.columns if c.startswith("auc_") and c.endswith("_fp32")]
    display(fp.groupby("arch", sort=False)[cols].agg(["mean", "std"]).round(3))

## 3. Compression sweep
`collapse_rate`: share of seeds where the model output (almost) nothing. `degraded_rate`: share of seeds with r² below 0.9× the FP32 model's. Thresholds are in `configs/default.yaml` and were chosen after seeing results (see README).

In [ ]:
tab = load("compression_table.csv")
if tab is not None:
    cols = ["arch", "bits", "sparsity", "n_seeds", "nnz_params", "total_weight_bits", "r2_mean",
            "collapse_rate", "degraded_rate", "auc_pooled_mean", "auc_pooled_std", "d_trivial_pooled_mean",
            "drop_pooled_mean", "drop_pooled_sem", "pareto", "pareto_full_quality"]
    display(tab.sort_values("total_weight_bits")[[c for c in cols if c in tab.columns]].round(3).reset_index(drop=True))

### Pareto plot (weight bits vs pooled AUC)
x = FP32 reference, grey x = collapsed configs, hollow = degraded r², red dotted line = trivial detector. Bars are the std over seeds.

In [ ]:
img = R / "pareto.png"
if img.exists():
    display(Image(filename=str(img)))
else:
    print("missing:", img.relative_to(ROOT))

### AUC against sparsity, per architecture and weight bit-width

In [ ]:
if tab is not None:
    archs = list(tab.arch.unique())
    fig, axes = plt.subplots(1, len(archs), figsize=(4.6 * len(archs), 3.8), sharey=True, squeeze=False)
    for ax, a in zip(axes[0], archs):
        for bits, g in tab[tab.arch == a].groupby("bits"):
            g = g.sort_values("sparsity")
            ax.errorbar(g.sparsity, g.auc_pooled_mean, yerr=g.auc_pooled_std.fillna(0), marker="o",
                        capsize=2, label=f"{bits}-bit weights")
            bad = g[g.collapse_rate > 0]
            ax.plot(bad.sparsity, bad.auc_pooled_mean, "kx", ms=9)
        if triv:
            ax.axhline(triv["pooled"], color="r", ls=":", lw=1)
        ax.set_title(a, fontsize=9)
        ax.set_xlabel("sparsity")
    axes[0][0].set_ylabel("pooled AUC (mean ± std)")
    axes[0][0].legend(fontsize=7)
    fig.suptitle("black x = collapsed in at least one seed; red dotted = trivial detector", fontsize=8)
    fig.tight_layout()
    plt.show()

### Reconstruction quality vs detection
If AUC tracked reconstruction quality, the points would rise to the right. They do not, which is why r² is reported next to AUC.

In [ ]:
if tab is not None:
    fig, ax = plt.subplots(figsize=(5.5, 4.2))
    for i, (a, g) in enumerate(tab.groupby("arch", sort=False)):
        ax.scatter(g.r2_mean, g.auc_pooled_mean, color=f"C{i}", label=a, alpha=0.8)
    if triv:
        ax.axhline(triv["pooled"], color="r", ls=":", lw=1)
    ax.set_xlabel("mean r² of the reconstruction (QCD test)")
    ax.set_ylabel("pooled AUC (mean over seeds)")
    ax.legend(fontsize=7)
    fig.tight_layout()
    plt.show()

### Per-class AUC minus the trivial detector (non-collapsed configs)
Positive = better than the trivial score. W and Z can improve while top jets stay worse.

In [ ]:
if tab is not None:
    v = tab[tab.collapse_rate == 0].sort_values("total_weight_bits")
    cc = [c for c in v.columns if c.startswith("d_trivial_") and c.endswith("_mean")]
    m = v[cc].to_numpy(dtype=float)
    fig, ax = plt.subplots(figsize=(5.5, 0.28 * len(v) + 1.5))
    im = ax.imshow(m, cmap="RdBu", vmin=-0.2, vmax=0.2, aspect="auto")
    ax.set_xticks(range(len(cc)))
    ax.set_xticklabels([c.replace("d_trivial_", "").replace("_mean", "") for c in cc])
    ax.set_yticks(range(len(v)))
    ax.set_yticklabels([f"{r.arch} {int(r.bits)}b s={r.sparsity:g}" for r in v.itertuples()], fontsize=6)
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    plt.show()

## 4. Collapse check on the saved seed-42 models
`r2` near 0 or `pct_out_zero` near 100 means the model outputs nothing; `d_*` is the AUC minus the trivial detector's.

In [ ]:
cc = load("collapse_check.csv")
if cc is not None:
    display(cc.round(3))

## 5. hls4ml agreement (C-simulation, seed-42 models)
`agrees` = at least 99% of outputs within one output LSB (1/32). AUC columns compare the Keras and hls4ml models on the same jets.

In [ ]:
h = load("hls_agreement.csv")
if h is not None:
    print(f"{int(h.agrees.sum())} of {len(h)} configs agree within 1 LSB for 99% of outputs")
    print("max |AUC difference| between Keras and hls4ml:", round(h.auc_diff.abs().max(), 4))
    display(h[~h.agrees][["arch", "bits", "sparsity", "max_abs_diff", "p99_abs_diff", "frac_within_atol",
                          "auc_pooled_keras", "auc_pooled_hls4ml"]].round(4))

## 6. Resource proxies (mode C) and latency
Multiplier counts, weight bits and BOPs from the saved seed-42 models. **No LUT/DSP numbers and no cycle counts exist**; latency was not assessed.

In [ ]:
rc = load("resource_estimates_C.csv")
if rc is not None:
    rc = rc.copy()
    rc["x_fewer_bits_than_fp32_baseline"] = (387 * 32) / rc.total_weight_bits
    display(rc[["arch", "bits", "sparsity", "n_mult", "total_weight_bits", "bops", "layer_depth",
                "x_fewer_bits_than_fp32_baseline"]].round(1))
lat = load("latency_report.csv")
if lat is not None:
    print("cycle estimates available:", int(lat.latency_best_cycles.notna().sum()), "of", len(lat))

## 7. Summary numbers

In [ ]:
if tab is not None:
    valid = tab[tab.collapse_rate == 0]
    print("configs:", len(tab), "| collapsed in >=1 seed:", int((tab.collapse_rate > 0).sum()),
          "| full quality:", int(tab.full_quality.sum()) if "full_quality" in tab else "n/a")
    if triv:
        print("non-collapsed configs beating the trivial detector (pooled):",
              int((valid.d_trivial_pooled_mean > 0).sum()), "of", len(valid))
    ok = valid[(valid.param_reduction_vs_baseline >= 0.6) & (valid.drop_pooled_mean < 0.02)]
    print("non-collapsed, >=60% fewer params and mean pooled drop < 0.02:", len(ok))
    print("Pareto points:", int(tab.pareto.sum()), "| full-quality only:", int(tab.pareto_full_quality.sum()))